In [2]:
import duckdb
import pandas as pd

# Connect to a local database file
con = duckdb.connect("../data/processed/provider_quality.db")

# Load raw CSVs into DuckDB tables
con.execute("""
    CREATE OR REPLACE TABLE hospitals AS
    SELECT * FROM read_csv_auto('../data/raw/hospitals.csv')
""")

con.execute("""
    CREATE OR REPLACE TABLE complications AS
    SELECT * FROM read_csv_auto('../data/raw/complications.csv')
""")

# Quick check
print("hospitals:", con.execute("SELECT COUNT(*) FROM hospitals").fetchone()[0])
print("complications:", con.execute("SELECT COUNT(*) FROM complications").fetchone()[0])

hospitals: 5432
complications: 95840


In [3]:
# What does the hospitals table look like?
con.execute("SELECT * FROM hospitals LIMIT 3").df()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,<NA>,15,15,<NA>,10,10,<NA>
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,<NA>,15,15,<NA>,10,10,<NA>
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,<NA>,15,15,<NA>,10,9,<NA>


In [4]:
# What does the complications table look like?
con.execute("SELECT * FROM complications LIMIT 3").df()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Measure ID,Measure Name,Compared to National,Denominator,Score,Lower Estimate,Higher Estimate,Footnote,Start Date,End Date
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,COMP_HIP_KNEE,Rate of complications for hip/knee replacement...,No Different Than the National Rate,27,3.2,1.7,5.9,None,2021-01-04,03/31/2024
1,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,No Different Than the National Rate,1835,4.5,2.6,7.4,None,2023-01-07,06/30/2024
2,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,MORT_30_AMI,Death rate for heart attack patients,No Different Than the National Rate,270,11.4,9.1,14.3,None,2021-01-07,06/30/2024


In [5]:
# What quality measures are available?
con.execute("""
    SELECT 
        "Measure ID",
        "Measure Name",
        COUNT(DISTINCT "Facility ID") as num_hospitals
    FROM complications
    GROUP BY "Measure ID", "Measure Name"
    ORDER BY num_hospitals DESC
""").df()

,Measure ID,Measure Name,num_hospitals
0,MORT_30_COPD,Death rate for COPD patients,4792
1,PSI_09,Postoperative hemorrhage or hematoma rate,4792
2,PSI_06,Iatrogenic pneumothorax rate,4792
3,PSI_04,Death rate among surgical inpatients with seri...,4792
4,PSI_15,Abdominopelvic accidental puncture or lacerati...,4792
5,PSI_13,Postoperative sepsis rate,4792
6,PSI_90,CMS Medicare PSI 90: Patient safety and advers...,4792
7,Hybrid_HWM,Hybrid Hospital-Wide All-Cause Risk Standardiz...,4792
8,MORT_30_STK,Death rate for stroke patients,4792
9,MORT_30_AMI,Death rate for heart attack patients,4792


In [6]:
# How many hospitals perform worse than national average for heart attack mortality?
con.execute("""
    SELECT 
        "Compared to National",
        COUNT(*) as num_hospitals,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM complications
    WHERE "Measure ID" = 'MORT_30_AMI'
      AND "Compared to National" IS NOT NULL
    GROUP BY "Compared to National"
    ORDER BY num_hospitals DESC
""").df()

,Compared to National,num_hospitals,pct
0,No Different Than the National Rate,1910,39.9
1,Number of Cases Too Small,1796,37.5
2,Not Available,1044,21.8
3,Better Than the National Rate,29,0.6
4,Worse Than the National Rate,13,0.3


In [7]:
# Which hospitals are worse than national average for heart attack mortality?
con.execute("""
    SELECT 
        c."Facility ID",
        c."Facility Name",
        h."State",
        h."Hospital Type",
        c."Score",
        c."Denominator"
    FROM complications c
    JOIN hospitals h 
        ON c."Facility ID" = h."Facility ID"
    WHERE c."Measure ID" = 'MORT_30_AMI'
      AND c."Compared to National" = 'Worse Than the National Rate'
    ORDER BY CAST(c."Score" AS FLOAT) DESC
""").df()

,Facility ID,Facility Name,State,Hospital Type,Score,Denominator
0,010039,HUNTSVILLE HOSPITAL,AL,Acute Care Hospitals,17.1,541
1,450133,MIDLAND MEMORIAL HOSPITAL,TX,Acute Care Hospitals,16.7,131
2,250082,DELTA HEALTH SYSTEM - THE MEDICAL CENTER,MS,Acute Care Hospitals,16.5,116
3,340143,CATAWBA VALLEY MEDICAL CENTER,NC,Acute Care Hospitals,16.2,114
4,440228,SAINT FRANCIS BARTLETT MEDICAL CENTER,TN,Acute Care Hospitals,16.1,148
5,050107,MARIAN REGIONAL MEDICAL CENTER,CA,Acute Care Hospitals,15.8,243
6,170020,HUTCHINSON REGIONAL MEDICAL CENTER INC,KS,Acute Care Hospitals,15.8,181
7,100157,LAKELAND REGIONAL MEDICAL CENTER,FL,Acute Care Hospitals,15.7,316
8,340070,ALAMANCE REGIONAL MEDICAL CENTER,NC,Acute Care Hospitals,15.7,125
9,440035,TENNOVA HEALTHCARE - CLARKSVILLE,TN,Acute Care Hospitals,15.5,184


## Finding 1: Hospitals Worse Than National Average for Heart Attack Mortality

Of 4,792 hospitals with data on 30-day heart attack mortality (`MORT_30_AMI`), 
only **13 hospitals (0.3%)** perform statistically worse than the national rate.

The `Score` column represents the 30-day mortality rate — the percentage of 
patients who die within 30 days of a heart attack admission. The national 
average is approximately 13%. These 13 hospitals range from 14.4% to 17.1%, 
meaning up to **4 additional deaths per 100 patients** compared to the national benchmark.

Note: 37.5% of hospitals had too few cases to evaluate, and 21.8% had no data 
available — a critical consideration for any provider recommendation model.

**Key SQL concepts used:** `JOIN`, `WHERE` filter, `CAST`, window function (`OVER()`).

In [9]:
# Build a composite quality score per hospital
# Using mortality measures only (lower score = better for these)
con.execute("""
    WITH mortality_measures AS (
        SELECT 
            c."Facility ID",
            c."Facility Name",
            c."Measure ID",
            CAST(c."Score" AS FLOAT) as score,
            CAST(c."Denominator" AS INTEGER) as denominator
        FROM complications c
        JOIN hospitals h ON c."Facility ID" = h."Facility ID"
        WHERE c."Measure ID" IN ('MORT_30_AMI', 'MORT_30_HF', 'MORT_30_PN', 
                                  'MORT_30_STK', 'MORT_30_COPD')
          AND c."Score" NOT IN ('Not Available', 'Not Applicable')
          AND c."Denominator" NOT IN ('Not Available', 'Not Applicable')
    ),
    hospital_scores AS (
        SELECT
            "Facility ID",
            "Facility Name",
            COUNT(DISTINCT "Measure ID")   as measures_available,
            ROUND(AVG(score), 2)           as avg_mortality_rate,
            ROUND(MIN(score), 2)           as best_measure,
            ROUND(MAX(score), 2)           as worst_measure,
            SUM(denominator)               as total_patients
        FROM mortality_measures
        GROUP BY "Facility ID", "Facility Name"
        HAVING COUNT(DISTINCT "Measure ID") >= 4
    ),
    ranked AS (
        SELECT 
            hs.*,
            h."State",
            RANK() OVER (ORDER BY avg_mortality_rate ASC) as quality_rank
        FROM hospital_scores hs
        JOIN hospitals h ON hs."Facility ID" = h."Facility ID"
    )
    SELECT *
    FROM ranked
    ORDER BY quality_rank
    LIMIT 20
""").df()

,Facility ID,Facility Name,measures_available,avg_mortality_rate,best_measure,worst_measure,total_patients,State,quality_rank
0,330214,NYU LANGONE HOSPITALS,5,6.96,5.5,8.8,8496.0,NY,1
1,05114F,VA SAN DIEGO HEALTHCARE SYSTEM,4,7.55,5.0,9.5,1227.0,CA,2
2,14003F,JESSE BROWN VA MEDICAL CENTER - VA CHICAGO HEA...,4,7.67,5.0,10.7,1272.0,IL,3
3,45066F,DALLAS VA MEDICAL CENTER (VA NORTH TEXAS HEALT...,4,7.73,5.7,9.6,1931.0,TX,4
4,11029F,DECATUR (ATLANTA) VA MEDICAL CENTER,4,7.88,5.4,12.0,1435.0,GA,5
5,09002F,WASHINGTON DC VA MEDICAL CENTER,4,8.25,4.5,11.5,1004.0,DC,6
6,220119,BRIGHAM AND WOMEN FAULKNER HOSPITAL,5,8.28,5.6,11.2,1153.0,MA,7
7,050625,CEDARS-SINAI MEDICAL CENTER,5,8.32,5.9,10.7,4426.0,CA,8
8,050135,SOUTHERN CALIFORNIA HOSPITAL AT HOLLYWOOD,5,8.36,5.8,11.9,1243.0,CA,9
9,05128F,VA GREATER LOS ANGELES HEALTHCARE SYSTEM,4,8.48,5.9,11.3,1108.0,CA,10


## Finding 2: Composite Mortality Quality Score

To rank hospitals across multiple conditions, we built a composite score 
averaging 5 mortality measures: heart attack, heart failure, pneumonia, 
stroke, and COPD — restricted to hospitals reporting at least 4 of the 5.

**Top performers:** NYU Langone (avg 6.96%), VA San Diego (7.55%), 
Brigham and Women's Faulkner (8.28%), Cedars-Sinai (8.32%), 
Mayo Clinic Rochester (8.84%), and Johns Hopkins (8.84%).

The national average for these measures is ~13%, so top-ranked hospitals 
show roughly **4-6 fewer deaths per 100 patients** than the national benchmark.

**Key SQL concepts used:** CTEs (`WITH`), `HAVING`, `RANK() OVER()`, 
`AVG/MIN/MAX` aggregations, data type casting.

In [13]:
con.execute("""
    WITH mortality_measures AS (
        SELECT 
            c."Facility ID",
            h."State",
            CAST(c."Score" AS FLOAT) as score,
            CAST(c."Denominator" AS INTEGER) as denominator
        FROM complications c
        JOIN hospitals h ON c."Facility ID" = h."Facility ID"
        WHERE c."Measure ID" IN ('MORT_30_AMI', 'MORT_30_HF', 'MORT_30_PN', 
                                  'MORT_30_STK', 'MORT_30_COPD')
          AND c."Score" NOT IN ('Not Available', 'Not Applicable')
          AND c."Denominator" NOT IN ('Not Available', 'Not Applicable')
    ),
    state_scores AS (
        SELECT
            "State",
            COUNT(DISTINCT "Facility ID")                        as num_hospitals,
            ROUND(SUM(score * denominator) / SUM(denominator), 2) as weighted_avg_mortality,
            ROUND(CAST(MIN(score) AS DECIMAL(10,2)), 2) as best_hospital_score,
            ROUND(CAST(MAX(score) AS DECIMAL(10,2)), 2) as worst_hospital_score
        FROM mortality_measures
        GROUP BY "State"
        HAVING COUNT(DISTINCT "Facility ID") >= 10
    )
    SELECT
        *,
        RANK() OVER (ORDER BY weighted_avg_mortality ASC) as state_rank
    FROM state_scores
    ORDER BY state_rank
    LIMIT 15
""").df()

,State,num_hospitals,weighted_avg_mortality,best_hospital_score,worst_hospital_score,state_rank
0,MA,53,11.46,5.6,21.2,1
1,MN,79,11.63,5.9,22.7,2
2,NY,143,11.65,5.5,26.5,3
3,IL,154,12.03,5.0,27.7,4
4,PA,135,12.05,5.7,23.1,5
5,CT,26,12.11,6.5,18.9,6
6,OH,136,12.13,5.8,24.8,7
7,CA,276,12.21,5.0,25.1,8
8,FL,173,12.38,5.8,24.4,9
9,NJ,62,12.40,6.3,22.8,10


## Finding 3: State-Level Quality Rankings (Weighted by Patient Volume)

Aggregating across 5 mortality measures weighted by patient volume, 
**Massachusetts ranks #1** among states with at least 10 reporting hospitals 
(weighted avg mortality: 11.46%).

Weighting by `Denominator` (number of patients) matters — using a simple 
average, NJ ranked #3; with weighted average it drops to #10, because its 
high-volume hospitals perform worse than its small ones.

**Top 5 states:** MA (11.46%), MN (11.63%), NY (11.65%), IL (12.03%), PA (12.05%).

Notable: New York ranks #3 at the state level despite having the 
top-ranked individual hospital (NYU Langone, 6.96%), reflecting high 
within-state variability.

**Key SQL concept:** Weighted average as `SUM(score * denominator) / SUM(denominator)` 
vs simple `AVG(score)` — critical when observations have different sample sizes.

In [14]:
# Does hospital ownership type affect quality outcomes?
con.execute("""
    WITH mortality_measures AS (
        SELECT 
            c."Facility ID",
            h."Hospital Ownership",
            CAST(c."Score" AS FLOAT)       as score,
            CAST(c."Denominator" AS INTEGER) as denominator
        FROM complications c
        JOIN hospitals h ON c."Facility ID" = h."Facility ID"
        WHERE c."Measure ID" IN ('MORT_30_AMI', 'MORT_30_HF', 'MORT_30_PN', 
                                  'MORT_30_STK', 'MORT_30_COPD')
          AND c."Score" NOT IN ('Not Available', 'Not Applicable')
          AND c."Denominator" NOT IN ('Not Available', 'Not Applicable')
    )
    SELECT
        "Hospital Ownership",
        COUNT(DISTINCT "Facility ID")                                    as num_hospitals,
        ROUND(SUM(score * denominator) / SUM(denominator), 2)           as weighted_avg_mortality,
        ROUND(MIN(score), 2)                                             as best_score,
        ROUND(MAX(score), 2)                                             as worst_score,
        RANK() OVER (ORDER BY SUM(score * denominator) / SUM(denominator) ASC) as quality_rank
    FROM mortality_measures
    GROUP BY "Hospital Ownership"
    ORDER BY quality_rank
""").df()

,Hospital Ownership,num_hospitals,weighted_avg_mortality,best_score,worst_score,quality_rank
0,Veterans Health Administration,121,9.46,4.5,18.500000,1
1,Physician,15,11.82,7.6,18.200001,2
2,Government - State,42,12.03,6.1,28.200001,3
3,Government - Federal,15,12.25,7.3,20.900000,4
4,Voluntary non-profit - Other,283,12.63,5.5,28.500000,5
5,Voluntary non-profit - Private,1799,12.69,5.1,26.500000,6
6,Voluntary non-profit - Church,240,12.75,5.8,24.400000,7
7,Proprietary,541,13.21,5.2,27.500000,8
8,Government - Hospital District or Authority,344,13.49,5.3,27.700001,9
9,Tribal,8,13.78,7.4,18.299999,10


## Finding 4: Hospital Ownership Type and Quality Outcomes

The most surprising finding: **Veterans Health Administration hospitals rank #1** 
with a weighted average mortality rate of 9.46% — nearly 3 points below the 
national average — outperforming all private, nonprofit, and other government systems.

**Ownership quality ranking (groups with 40+ hospitals):**
- 🥇 Veterans Health Administration: 9.46% (121 hospitals)
- 🥈 Government - State: 12.03% (42 hospitals)
- 🥉 Voluntary non-profit - Other: 12.63% (283 hospitals)
- Voluntary non-profit - Private: 12.69% (1,799 hospitals)
- Voluntary non-profit - Church: 12.75% (240 hospitals)
- ❌ Proprietary (for-profit): 13.21% (541 hospitals)
- ❌ Government - Hospital District: 13.49% (344 hospitals)
- ❌ Government - Local: 14.24% (269 hospitals)

Physician-owned (11.82%) and Tribal (13.78%) hospitals are excluded from 
the ranking — with only 15 and 8 hospitals respectively, the samples are 
too small for reliable comparison.

**Caveat:** This is observational data. VA hospitals may serve a different 
patient population and case mix, which could partially explain the difference. 
A fair comparison would require risk adjustment — a natural next step for this analysis.

**Key SQL concept:** `RANK() OVER()` applied directly on an aggregated expression 
without a separate CTE.